# Chimera State Detection

A **chimera state** is the coexistence of coherent (phase-locked) and
incoherent (drifting) oscillator populations in an otherwise identical
network. First observed by Kuramoto & Battogtokh (2002), chimeras arise
in nonlocally coupled rings where nearest neighbors lock while distant
oscillators desynchronize.

Detection uses a **local order parameter** R_i:
```
R_i = |mean(exp(i * (theta_j - theta_i)))| for neighbors j of i
```
- R_i > 0.7: oscillator i is in the coherent group
- R_i < 0.3: oscillator i is in the incoherent group
- Between: boundary region

The **chimera_index** measures the fraction of boundary oscillators --
a higher value signals a more pronounced chimera state.

In [ ]:
import numpy as np

from scpn_phase_orchestrator.monitor.chimera import detect_chimera
from scpn_phase_orchestrator.upde.engine import UPDEEngine
from scpn_phase_orchestrator.upde.order_params import compute_order_parameter

In [ ]:
# Set up 16 oscillators in a ring with nonlocal coupling.
# Chimera-promoting topology: couple each oscillator to its R nearest
# neighbors on each side (nonlocal, not all-to-all).
N_OSC = 16
R_NEIGHBORS = 4  # couple to 4 neighbors on each side
K_STRENGTH = 0.6
TWO_PI = 2.0 * np.pi

# Build nonlocal ring coupling
knm = np.zeros((N_OSC, N_OSC))
for i in range(N_OSC):
    for dr in range(1, R_NEIGHBORS + 1):
        j_right = (i + dr) % N_OSC
        j_left = (i - dr) % N_OSC
        # Coupling decays with distance
        strength = K_STRENGTH * np.exp(-0.3 * dr)
        knm[i, j_right] = strength
        knm[i, j_left] = strength

# Identical natural frequencies (chimeras require near-identical omegas)
omegas = np.ones(N_OSC) * 1.0
alpha = np.zeros((N_OSC, N_OSC))

# Initial conditions: half synchronized, half random -- seeds chimera
rng = np.random.default_rng(17)
phases = np.zeros(N_OSC)
phases[:8] = 0.0 + rng.normal(0, 0.05, 8)  # coherent cluster
phases[8:] = rng.uniform(0, TWO_PI, 8)  # incoherent cluster

print(f"Network: {N_OSC} oscillators, {R_NEIGHBORS} neighbors each side")
print(f"Coupling strength: {K_STRENGTH}")
print("Initial coherent: osc 0-7, incoherent: osc 8-15")

In [ ]:
# Run UPDEEngine, detect chimera at each step
DT = 0.01
STEPS = 3000
engine = UPDEEngine(N_OSC, dt=DT)

chimera_idx_hist = []
R_hist = []
n_coherent_hist = []
n_incoherent_hist = []
phase_snapshots = []

for step in range(STEPS):
    phases = engine.step(phases, omegas, knm, 0.0, 0.0, alpha)

    cs = detect_chimera(phases, knm)
    chimera_idx_hist.append(cs.chimera_index)
    n_coherent_hist.append(len(cs.coherent_indices))
    n_incoherent_hist.append(len(cs.incoherent_indices))

    R, _ = compute_order_parameter(phases)
    R_hist.append(R)

    # Save snapshots for visualization
    if step % 500 == 0:
        phase_snapshots.append((step, phases.copy(), cs))

print(f"Simulation complete: {STEPS} steps")
print(f"Final chimera_index: {chimera_idx_hist[-1]:.4f}")
print(f"Final R: {R_hist[-1]:.4f}")

In [ ]:
# Plot chimera_index and R over time
try:
    import matplotlib.pyplot as plt

    t = np.arange(STEPS) * DT
    fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

    axes[0].plot(t, chimera_idx_hist, linewidth=0.8, color="tab:red")
    axes[0].set_ylabel("Chimera index")
    axes[0].set_title("Chimera Detection: 16 Oscillators, Nonlocal Ring")

    axes[1].plot(t, R_hist, linewidth=0.8, color="tab:blue")
    axes[1].set_ylabel("R (global)")
    axes[1].axhline(
        0.7, color="gray", linestyle="--", linewidth=0.5, label="coherent threshold"
    )
    axes[1].axhline(
        0.3, color="gray", linestyle=":", linewidth=0.5, label="incoherent threshold"
    )
    axes[1].legend(fontsize=8)

    axes[2].plot(t, n_coherent_hist, linewidth=0.8, label="coherent", color="tab:green")
    axes[2].plot(
        t, n_incoherent_hist, linewidth=0.8, label="incoherent", color="tab:orange"
    )
    axes[2].set_ylabel("Count")
    axes[2].set_xlabel("Time (s)")
    axes[2].legend(fontsize=8)

    fig.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed, skipping plot")

In [ ]:
# Show coherent vs incoherent oscillator groups at snapshots
for step, snap_phases, cs in phase_snapshots:
    print(f"\n--- Step {step} ---")
    print(f"  Coherent oscillators:   {cs.coherent_indices}")
    print(f"  Incoherent oscillators: {cs.incoherent_indices}")
    print(f"  Chimera index: {cs.chimera_index:.4f}")
    # Phase spread within each group
    if cs.coherent_indices:
        coh_phases = snap_phases[cs.coherent_indices]
        coh_spread = np.std(coh_phases % TWO_PI)
        print(f"  Coherent phase spread (std): {coh_spread:.4f} rad")
    if cs.incoherent_indices:
        inc_phases = snap_phases[cs.incoherent_indices]
        inc_spread = np.std(inc_phases % TWO_PI)
        print(f"  Incoherent phase spread (std): {inc_spread:.4f} rad")

# Final polar plot of oscillator phases colored by group
try:
    import matplotlib.pyplot as plt

    final_cs = detect_chimera(phases, knm)
    fig, ax = plt.subplots(subplot_kw={"projection": "polar"}, figsize=(6, 6))

    for i in range(N_OSC):
        if i in final_cs.coherent_indices:
            color, label = "tab:green", "coherent"
        elif i in final_cs.incoherent_indices:
            color, label = "tab:red", "incoherent"
        else:
            color, label = "tab:gray", "boundary"
        ax.plot(phases[i] % TWO_PI, 1.0, "o", color=color, markersize=10)

    # Legend (deduplicate)
    from matplotlib.lines import Line2D

    handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="tab:green",
            markersize=10,
            label="Coherent",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="tab:red",
            markersize=10,
            label="Incoherent",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="tab:gray",
            markersize=10,
            label="Boundary",
        ),
    ]
    ax.legend(handles=handles, loc="upper right", fontsize=8)
    ax.set_title("Final Phase Distribution", pad=15)
    ax.set_ylim(0, 1.3)
    fig.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed, skipping plot")